In [2]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModel, Trainer, TrainingArguments
from datasets import load_dataset

from pydantic import BaseModel
import faiss
import nltk
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.rouge_score import rouge_n
import bleach
import re
import spacy
from opacus import PrivacyEngine
from sklearn.metrics import f1_score
from typing import List, Dict
import os

# Download required NLTK data
nltk.download('punkt')

# Initialize FastAPI app
app = FastAPI(title="Adaptive Emotion-Guided Healthcare Chatbot")

# Initialize SpaCy for anonymization
nlp = spacy.load("en_core_web_sm")

# Load datasets
def load_datasets():
    """Load MedQuAD, PubMedQA, and GoEmotions datasets."""
    # Load PubMedQA via Hugging Face
    pubmedqa = load_dataset("pubmed_qa", "pqa_labeled", split="train")
    pubmedqa_df = pd.DataFrame({
        "question": pubmedqa["question"],
        "answer": [ctx["long_answer"] for ctx in pubmedqa["context"]],
        "context": [ctx["contexts"] for ctx in pubmedqa["context"]]
    })

    # Load MedQuAD (assumes CSV file downloaded from https://github.com/abachaa/MedQuAD)
    # Replace with actual path after downloading
    medquad_path = "data/MedQuAD.csv"  # Update with actual path
    if os.path.exists(medquad_path):
        medquad_df = pd.read_csv(medquad_path)
    else:
        # Fallback to sample for demo
        medquad_df = pd.DataFrame({
            "question": ["What are the symptoms of diabetes?", "How to treat a minor burn?", "I’m worried about my heart condition."],
            "answer": ["Increased thirst, frequent urination, fatigue, blurred vision.", "Cool under running water, apply aloe vera, cover with bandage.", "Consult a cardiologist, consider stress management techniques."]
        })
    
    # Simulate emotional annotations (GoEmotions not directly loaded for simplicity; use pretrained model instead)
    medquad_df["emotion"] = medquad_df["question"].apply(
        lambda x: "anxious" if any(kw in x.lower() for kw in ["worried", "anxious", "scared"]) else "neutral"
    )
    
    return medquad_df, pubmedqa_df

# Data cleaning and preprocessing
def clean_text(text: str) -> str:
    """Clean text by removing special characters and normalizing spaces."""
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace("diabetes mellitus", "diabetes")  # Normalize medical terms
    return text

def anonymize_text(text: str) -> str:
    """Anonymize sensitive data using SpaCy NER."""
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "DATE", "GPE"]:
            text = text.replace(ent.text, f"[{ent.label_}]")
    return text

def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess dataset for training and RAG."""
    df['question'] = df['question'].apply(clean_text).apply(anonymize_text)
    df['answer'] = df['answer'].apply(clean_text).apply(anonymize_text)
    # Simulate diversity check (e.g., balance across demographics)
    # In production, stratify by age/gender if metadata available
    return df

# Load and preprocess datasets
medquad_df, pubmedqa_df = load_datasets()
medquad_df = preprocess_dataset(medquad_df)
knowledge_base = pubmedqa_df["context"].apply(lambda x: " ".join(x)).tolist()[:100]  # Limit for demo

# Initialize BioBERT and emotion detection models
biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
biobert_qa_model = AutoModelForQuestionAnswering.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
biobert_embedding_model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
emotion_tokenizer = AutoTokenizer.from_pretrained("monologg/bert-base-cased-goemotions-ekman")
emotion_model = AutoModel.from_pretrained("monologg/bert-base-cased-goemotions-ekman")  # Pretrained GoEmotions model

# Initialize FAISS index for RAG
dimension = 768  # BioBERT embedding size
index = faiss.IndexFlatL2(dimension)

# Generate embeddings for knowledge base
def create_knowledge_base_embeddings():
    """Convert knowledge base documents to BioBERT embeddings and store in FAISS."""
    embeddings = []
    for doc in knowledge_base:
        inputs = biobert_tokenizer(doc, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = biobert_embedding_model(**inputs).last_hidden_state[:, 0, :]  # CLS token
        embeddings.append(outputs.squeeze().numpy())
    embeddings = np.array(embeddings)
    index.add(embeddings)
    return embeddings

knowledge_base_embeddings = create_knowledge_base_embeddings()

# Simulated fine-tuning for BioBERT
def fine_tune_biobert(df: pd.DataFrame):
    """Simulate fine-tuning BioBERT with empathetic prompts and fairness-aware loss."""
    training_args = TrainingArguments(
        output_dir="./biobert_finetuned",
        num_train_epochs=1,
        per_device_train_batch_size=8,
        logging_steps=10,
        save_steps=100,
        evaluation_strategy="steps"
    )
    trainer = Trainer(
        model=biobert_qa_model,
        args=training_args,
        train_dataset=df  # Requires proper dataset format in production
    )
    # Add differential privacy
    privacy_engine = PrivacyEngine()
    model, optimizer, train_dataloader = privacy_engine.make_private(
        module=biobert_qa_model,
        optimizer=trainer.optimizer,
        data_loader=trainer.get_train_dataloader(),
        max_grad_norm=1.0
    )
    print("Simulated fine-tuning completed with differential privacy.")

# Emotion detection using pretrained GoEmotions model
def detect_emotion(query: str) -> str:
    """Detect emotion using pretrained GoEmotions model."""
    inputs = emotion_tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = emotion_model(**inputs)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item()
    # Map to simplified emotions (Ekman categories: anger, disgust, fear, joy, sadness, surprise, neutral)
    emotion_map = {0: "anger", 1: "disgust", 2: "fear", 3: "joy", 4: "sadness", 5: "surprise", 6: "neutral"}
    return emotion_map.get(predicted_class, "neutral")

# Input sanitization
def sanitize_input(query: str) -> str:
    """Sanitize user input to prevent injection attacks."""
    sanitized = bleach.clean(query, tags=[], strip=True)
    if re.search(r"[<>{};]", sanitized):
        raise HTTPException(status_code=400, detail="Invalid input detected")
    return sanitized

# RAG pipeline
def rag_pipeline(query: str) -> str:
    """Retrieve relevant documents and generate response using BioBERT."""
    inputs = biobert_tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        query_embedding = biobert_embedding_model(**inputs).last_hidden_state[:, 0, :].squeeze().numpy()
    
    # Retrieve top-k documents
    k = 1
    distances, indices = index.search(np.array([query_embedding]), k)
    retrieved_doc = knowledge_base[indices[0][0]]
    
    # Generate response
    inputs = biobert_tokenizer(query, retrieved_doc, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = biobert_qa_model(**inputs)
        start = torch.argmax(outputs.start_logits)
        end = torch.argmax(outputs.end_logits) + 1
        answer = biobert_tokenizer.decode(inputs["input_ids"][0][start:end])
    
    return answer

# Emotion-guided response generation
def generate_empathic_response(query: str, answer: str, emotion: str) -> str:
    """Generate empathetic response based on detected emotion."""
    if emotion in ["fear", "sadness"]:
        return f"I’m here to support you through this concern. {answer} Would you like additional resources or guidance?"
    return answer

# Bias mitigation check (simulated demographic parity)
def check_demographic_parity(responses: List[str], demographics: List[str]) -> float:
    """Simulate demographic parity check for fairness."""
    # Placeholder: Assume equal response quality across demographics
    return 0.95  # Simulated parity score

# Evaluation function
def evaluate_chatbot(queries: List[str], references: List[str], demographics: List[str]) -> Dict:
    """Evaluate chatbot performance, empathy, and fairness."""
    bleu_scores = []
    rouge_scores = []
    emotions = []
    true_labels = [detect_emotion(q) for q in queries]  # Simulated ground truth
    
    for query, ref in zip(queries, references):
        answer = rag_pipeline(query)
        emotion = detect_emotion(query)
        bleu = sentence_bleu([ref.split()], answer.split())
        rouge = rouge_n(ref.split(), answer.split(), n=1)
        bleu_scores.append(bleu)
        rouge_scores.append(rouge)
        emotions.append(emotion)
    
    f1 = f1_score(true_labels, emotions, average='weighted')
    parity = check_demographic_parity([rag_pipeline(q) for q in queries], demographics)
    
    return {
        "avg_bleu": np.mean(bleu_scores),
        "avg_rouge": np.mean(rouge_scores),
        "f1_emotion": f1,
        "demographic_parity": parity
    }

# Active learning for knowledge base update
def update_knowledge_base(new_doc: str):
    """Add new document to knowledge base and update FAISS index."""
    knowledge_base.append(new_doc)
    inputs = biobert_tokenizer(new_doc, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        embedding = biobert_embedding_model(**inputs).last_hidden_state[:, 0, :].squeeze().numpy()
    index.add(np.array([embedding]))

# Pydantic model for request
class QueryRequest(BaseModel):
    query: str



# Simulated ablation study
def ablation_study():
    """Simulate ablation study for DEGKR components."""
    queries = medquad_df['question'].tolist()[:10]
    references = medquad_df['answer'].tolist()[:10]
    demographics = ["male", "female", "other"] * (len(queries) // 3 + 1)
    results = {
        "full_degkr": evaluate_chatbot(queries, references, demographics),
        "no_emotion": {"avg_bleu": 0.80, "f1_emotion": 0.0},  # Simulated
        "no_rag": {"avg_bleu": 0.70, "f1_emotion": 0.85},  # Simulated
        "no_bias_mitigation": {"demographic_parity": 0.60}  # Simulated
    }
    return results

if __name__ == "__main__":
    # Simulate fine-tuning
    fine_tune_biobert(medquad_df)
    
    # Run evaluation
    eval_results = evaluate_chatbot(
        queries=medquad_df['question'].tolist()[:10],
        references=medquad_df['answer'].tolist()[:10],
        demographics=["male", "female", "other"] * (len(medquad_df) // 3 + 1)
    )
    print("Evaluation Results:", eval_results)
    
    # Run ablation study
    ablation_results = ablation_study()
    print("Ablation Study Results:", ablation_results)
    
    # Start FastAPI server
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

ModuleNotFoundError: No module named 'nltk.translate.rouge_score'

# Adaptive Emotion-Guided Healthcare Chatbot

In [2]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.8 MB/s eta 0:00:00a 0:00:01


In [13]:
!pip install datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
from datasets import load_dataset

ds = load_dataset("keivalya/MedQuad-MedicalQnADataset")
# Inspect one sample
print(ds["train"][0])


{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


In [20]:
# Extract just the answers for indexing
docs = [sample["Answer"] for sample in ds["train"] if sample["Answer"]]


In [21]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load your model
model = SentenceTransformer("pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb")

# Use a subset to avoid memory overload (you can increase later)
docs_subset = docs[:1000]
embeddings = model.encode(docs_subset, show_progress_bar=True)

# Create FAISS index
index = faiss.IndexFlatL2(768)
index.add(np.array(embeddings))

# Save
faiss.write_index(index, "index.faiss")
np.save("doc_store.npy", np.array(docs_subset, dtype=object))


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [22]:
query = "How is asthma treated?"
query_embedding = model.encode([query])
D, I = index.search(np.array(query_embedding), k=3)

for idx in I[0]:
    print("🔎", docs_subset[idx])


🔎 Arachnoiditis remains a difficult condition to treat, and long-term outcomes are unpredictable. Most treatments for arachnoiditis are focused on pain relief and the improvement of symptoms that impair daily function. A regimen of pain management, physiotheraphy, exercise, and psychotheraphy is often recommended. Surgical intervention is controversial since the outcomes are generally poor and provide only short-term relief.
🔎 There is no cure for lupus. Treatment is symptomatic. With a combination of medication, rest, exercise, proper nutrition, and stress management, most individuals with lupus can often achieve remission or reduce their symptom levels. Medications used in the treatment of lupus may include aspirin and other nonsteroidal anti-inflammatory medications, antimalarials, corticosteroids, and immunosuppressive drugs.
🔎 Currently there is no cure for these disease syndromes.Medical care is directed at treating systemic conditions and improving the person's quality of life. 

In [50]:
from transformers import pipeline

# Re-define the generator with GPT-2
generator = pipeline("text-generation", model="gpt2")

# Use a structured and guided prompt
prompt = f"""Patient question: How is asthma treated?

Relevant medical information:
{context}

Response:"""

# Generate a response using GPT-2
response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    num_return_sequences=1,
    pad_token_id=50256,
    truncation=True
)[0]['generated_text']

# Show only the generated answer
print("🩺 Chatbot Response:\n")
print(response.split("Response:")[-1].strip())


Device set to use mps:0


🩺 Chatbot Response:

Patient responds to therapy with ease, and in many cases with great benefit. However, the most common response involves burning and may lead to severe allergic reaction.

For many patients there is limited clinical evidence to confirm the presence of Sjgren's syndrome, because many patients have mild to moderate asthma with little to no evidence of symptoms. There are no known effective treatments and several existing treatments are ineffective. In most cases some studies suggest that interferon alpha 

This response depends on a


In [23]:
!pip install transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [33]:
!pip install sacremoses

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 1.7 MB/s eta 0:00:00a 0:00:01


In [43]:
pip install trl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Attempting uninstall: datasets
    Found existing installation: datasets 2.16.1
    Uninstalling datasets-2.16.1:
      Successfully uninstalled datasets-2.16.1
Note: you may need to restart the kernel to use updated packages.


In [51]:
from datasets import load_dataset

# Load the MedQuAD dataset
dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset")

# Extract answers (you can filter for "asthma" later if needed)
docs = [entry['Answer'] for entry in dataset['train'] if entry['Answer']]


In [52]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load BioBERT model from sentence-transformers
model = SentenceTransformer("pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb")

# Limit documents for speed
docs_subset = docs[:5000]

# Convert text to embeddings
embeddings = model.encode(docs_subset, show_progress_bar=True)

# Create FAISS index
index = faiss.IndexFlatL2(768)
index.add(np.array(embeddings))

# Save index and docs
faiss.write_index(index, "index.faiss")
np.save("doc_store.npy", np.array(docs_subset, dtype=object))


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [53]:
# Example query
query = "How is asthma treated?"
query_embedding = model.encode([query])

# Search top 3 similar docs
D, I = index.search(np.array(query_embedding), k=3)

# Load documents
docs_subset = np.load("doc_store.npy", allow_pickle=True)
retrieved_docs = [docs_subset[idx] for idx in I[0]]

# Prepare context
context = "\n".join(retrieved_docs)


In [14]:
!ollama pull llama3
from ollama import Client

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
import requests
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- 1. Load Dataset ---
dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset")
docs = [entry['Answer'] for entry in dataset['train'] if entry['Answer']]

# --- 2. Encode and Index with FAISS ---
model = SentenceTransformer('pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb')
doc_embeddings = model.encode(docs, show_progress_bar=True)
faiss_index = faiss.IndexFlatL2(doc_embeddings.shape[1])
faiss_index.add(np.array(doc_embeddings))
np.save("docs.npy", docs)
faiss.write_index(faiss_index, "index.faiss")

# --- 3. Retrieve Similar Docs Based on Query ---
def retrieve_context(user_query, k=3):
    query_vec = model.encode([user_query])
    D, I = faiss_index.search(np.array(query_vec), k=k)
    return "\n".join([docs[idx] for idx in I[0]])

# --- 4. Query LLaMA via Ollama API ---
def query_llama(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3", "prompt": prompt, "stream": False}
    )
    return response.json()['response'].strip()

# --- 5. Main Chatbot Function ---
def careconnect_chat(user_question):
    context = retrieve_context(user_question)
    prompt = f"""You are CareConnect, a trusted medical assistant.

Answer the user's medical question truthfully using the information below. Do not answer questions unrelated to health.

### CONTEXT:
{context}

### USER QUESTION:
{user_question}

### RESPONSE:"""

    answer = query_llama(prompt)
    return answer

# --- 6. Example Usage ---
query = "How is asthma treated?"
response = careconnect_chat(query)
print("🩺 CareConnect Response:\n", response)


Batches:   0%|          | 0/513 [00:00<?, ?it/s]

🩺 CareConnect Response:
 I'm happy to help! As a trusted medical assistant, I can provide you with accurate information on how asthma is treated.

Asthma treatment typically involves two types of medications:

1. **Long-term control medications**: These medicines are taken daily to reduce inflammation in the airways and prevent symptoms from occurring. Examples include inhaled corticosteroids (such as fluticasone or budesonide), long-acting beta2-agonists (such as salmeterol or formoterol), and oral corticosteroids (in severe cases).
2. **Quick-relief medications**: These medicines are used to relieve sudden asthma symptoms, such as wheezing, coughing, or shortness of breath. Examples include inhaled albuterol (such as Ventolin) or levalbuterol (such as Xopenex).

In addition to medication, other treatment options may be recommended by your healthcare provider:

* **Peak flow monitoring**: This involves using a peak flow meter to measure the amount of air that can be exhaled from the l

In [16]:
def symptom_checker_bot(user_symptoms):
    context = retrieve_context(user_symptoms)
    prompt = f"""You are CareConnect, a cautious virtual assistant helping users understand symptoms. You are not a doctor.

Provide helpful advice based only on the context below. Include a disclaimer that this is not medical advice.

### CONTEXT:
{context}

### USER SYMPTOMS:
{user_symptoms}

### RESPONSE:"""
    return query_llama(prompt)


In [17]:
def prescription_explainer_bot(drug_name):
    context = retrieve_context(drug_name)
    prompt = f"""You are CareConnect, a virtual pharmacist assistant.

Explain the dosage, common side effects, and important interactions of the drug or prescription term given below, using only the provided context.

### CONTEXT:
{context}

### DRUG NAME / INSTRUCTION:
{drug_name}

### RESPONSE:"""
    return query_llama(prompt)


In [18]:
def health_literacy_tutor_bot(user_question):
    context = retrieve_context(user_question)
    prompt = f"""You are CareConnect, a patient-friendly health tutor.

Explain the concept in very simple, easy-to-understand language suitable for someone with no medical background.

### CONTEXT:
{context}

### QUESTION:
{user_question}

### RESPONSE:"""
    return query_llama(prompt)


In [20]:
symptom_checker_bot("I have chest pain and shortness of breath in the morning.")


"Hi there! I'm CareConnect, here to help you understand your symptoms. Based on what you've told me, you're experiencing chest pain and shortness of breath in the morning. These symptoms can be concerning, but it's important to remember that they could be caused by various factors.\n\nFrom a non-medical perspective, it's possible that your symptoms might be related to other conditions, such as respiratory issues or even primary myelofibrosis (which can present with chest pain and shortness of breath). However, without a thorough medical evaluation, I cannot determine the exact cause of your symptoms.\n\n**Important disclaimer:** I am not a doctor, and this is not medical advice. If you're experiencing persistent or severe symptoms like chest pain and shortness of breath, it's crucial to consult with a qualified healthcare professional for proper diagnosis and treatment.\n\nIn the meantime, here are some general suggestions that might help:\n\n1. Take slow, deep breaths: This can help a

In [21]:
prescription_explainer_bot("Metformin 500mg twice a day")


"Hello! I'm CareConnect, your virtual pharmacist assistant. I'd be happy to help you understand the dosage, common side effects, and important interactions of Metformin 500mg taken twice a day.\n\n**Dosage:** The recommended dosage for Metformin is 500mg taken two times a day.\n\n**Common Side Effects:**\n\nAs an oral diabetes medication that increases insulin production, Metformin can cause hypoglycemia (low blood sugar) when used alone or in combination with other diabetes medications. Other common side effects of Metformin may include:\n\n* Diarrhea\n* Nausea\n* Abdominal pain\n* Flatulence\n\n**Important Interactions:**\n\nWhen taken with other diabetes medications, such as insulin or pills that increase insulin production (like those listed earlier), Metformin can increase the risk of hypoglycemia. Additionally, if you're taking Metformin and experience symptoms like dizziness, shakiness, or confusion, it's essential to seek medical attention promptly.\n\nRemember to always follow

In [22]:
health_literacy_tutor_bot("What is chronic obstructive pulmonary disease?")

"Hi there! I'm CareConnect, your patient-friendly health tutor. I'll be happy to explain these concepts in simple terms.\n\nLet's start with tracheobronchopathia osteoplastica (TO). It's a condition that affects the airways and lungs. The good news is that even though there isn't a specific treatment for TO, doctors can help manage symptoms like recurrent infections and lung collapse. They may also suggest inhaled corticosteroids or even surgery to help relieve symptoms.\n\nNow, let's talk about bronchitis. Bronchitis is an inflammation of the airways that carry air to our lungs. It causes a cough that often brings up mucus, shortness of breath, wheezing, and chest tightness. There are two main types: acute and chronic. Chronic bronchitis is one type of COPD (chronic obstructive pulmonary disease).\n\nCOPD is when the airways become damaged and narrowed, making it harder to breathe. The inflamed airways produce a lot of mucus, which leads to coughing and difficulty breathing. Smoking i

## NewsLensBot : 
Uses AI to make news articles easier to understand and analyze for everyone.
- News can be hard to read (too much jargon, complex sentences).
- News can be biased (very positive/negative, opinionated).
- People want short, clear, and fair news.

In [1]:
!pip install transformers datasets textstat --quiet

#NutriFit Advisor: Personalized Diet & Fitness AI

### NewsLensBot  : It takes a news article (or any news text you give), and does several smart things, step by step:



In [13]:
from transformers import pipeline, BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import textstat
import re
import pandas as pd
import torch


In [5]:
from datasets import load_dataset

ds = load_dataset("abisee/cnn_dailymail", "3.0.0",split="test[:1%]")


In [6]:
print(f"Number of samples: {len(ds)}")

# Example sample
article = ds[0]['article']
human_summary = ds[0]['highlights']

Number of samples: 115


In [14]:
print("News Article (first 300 chars):", article[:300] + "...")
print("\n Human Summary:", human_summary)

News Article (first 300 chars): (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou...

 Human Summary: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [15]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
generated_summary = summarizer(article, max_length=130, min_length=30, do_sample=False)[0]['summary_text']

print("🤖 BART Summary:", generated_summary)


Device set to use mps:0


🤖 BART Summary: The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body.


In [16]:
# For demo: take a tiny subset for quick fine-tuning
small_train = load_dataset("cnn_dailymail", "3.0.0", split="train[:100]")
small_val = load_dataset("cnn_dailymail", "3.0.0", split="validation[:20]")

model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

def preprocess_function(examples):
    inputs = examples["article"]
    targets = examples["highlights"]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_val = small_val.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir="./bart_finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    logging_steps=5,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

trainer.train()

# Save the finetuned model
model.save_pretrained("./bart_finetuned")
tokenizer.save_pretrained("./bart_finetuned")


Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/opt/anaconda3/envs/gouthami/lib/python3.11/site-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.308900,1.027411


/opt/anaconda3/envs/gouthami/lib/python3.11/site-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('./bart_finetuned/tokenizer_config.json',
 './bart_finetuned/special_tokens_map.json',
 './bart_finetuned/vocab.json',
 './bart_finetuned/merges.txt',
 './bart_finetuned/added_tokens.json')

In [20]:
finetuned_summarizer = pipeline("summarization", model="./bart_finetuned")
finetuned_summary = finetuned_summarizer(article, max_length=130, min_length=30, do_sample=False)[0]['summary_text']
print("🧑‍🔬 Fine-Tuned BART Summary:", finetuned_summary)


Device set to use mps:0


🧑‍🔬 Fine-Tuned BART Summary: The Palestinian Authority officially becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. The court has opened a preliminary examination into the situation in the Palestinian territories, paving the way for possible investigations.


In [21]:
simplify_dict = {
    "approximately": "about", "individuals": "people", "terminate": "end",
    "utilize": "use", "assistance": "help", "demonstrate": "show",
    "commence": "start", "requirement": "need", "reside": "live"
}

def simplify_text(text, simplify_dict):
    pattern = re.compile(r'\b(' + '|'.join(simplify_dict.keys()) + r')\b', re.IGNORECASE)
    return pattern.sub(lambda m: simplify_dict[m.group(0).lower()], text)

simple_summary = simplify_text(generated_summary, simplify_dict)


In [23]:
from transformers import pipeline

# Create the zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define your topic categories (edit/add more if you want!)
candidate_labels = ["politics", "sports", "business", "health", "entertainment", "technology", "science", "world"]

# Run classification
result = classifier(simple_summary, candidate_labels)
print("Top topic:", result["labels"][0], "| Score:", result["scores"][0])
print("All topic scores:")
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.2f}")


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use mps:0


Top topic: politics | Score: 0.4508017599582672
All topic scores:
politics: 0.45
world: 0.27
business: 0.10
technology: 0.06
entertainment: 0.05
health: 0.03
science: 0.02
sports: 0.02


In [24]:
sentiment_analyzer = pipeline("sentiment-analysis", model="roberta-base")

def detect_bias(text):
    result = sentiment_analyzer(text)[0]
    label = result["label"]
    score = result["score"]
    if label == "POSITIVE" and score > 0.9:
        return "Potential positive bias"
    elif label == "NEGATIVE" and score > 0.9:
        return "Potential negative bias"
    else:
        return "Neutral"

bias_result = detect_bias(simple_summary)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use mps:0


In [25]:
original_grade = textstat.flesch_kincaid_grade(generated_summary)
simple_grade = textstat.flesch_kincaid_grade(simple_summary)


In [27]:
import pandas as pd

output_data = {
    "Human Summary": [human_summary],
    "Model Summary": [generated_summary],
    "Simplified": [simple_summary],
    "Topic": [result["labels"][0]],
    "Bias": [bias_result],
    "Orig Grade": [original_grade],
    "Simp Grade": [simple_grade]
}

df = pd.DataFrame(output_data)
df.T  # Transpose for better readability


,0
Human Summary,Membership gives the ICC jurisdiction over all...
Model Summary,The Palestinian Authority becomes the 123rd me...
Simplified,The Palestinian Authority becomes the 123rd me...
Topic,politics
Bias,Neutral
Orig Grade,13.138919
Simp Grade,13.138919


In [1]:
#sample input test
article = """
A day after Prime Minister Narendra Modi's fiery address to the nation on the success of Operation Sindoor launched by Indian armed forces against Pakistani terror infrastructure, the Pakistan government said it “rejects provocative and inflammatory assertions by the Indian PM”.Prime Minister Modi on Monday addressed the nation and lauded the Indian armed forces for successfully carrying out Operation Sindoor, which destroyed key terror infrastructure and eliminated dozens of terrorists, some of whom were “high-value”. Follow India-Pakistan news LIVE updates

In strong warnings to Pakistan, Modi said India has only paused retaliation against the terrorist and military bases of the country and has not ended it, adding that the ceasefire was first requested by Islamabad.

Modi also said “terror or talk can't go together, terror and trade cannot go hand-in-hand and that water and blood also cannot flow together”.  

Pakistan's foreign ministry, reacting to Modi's address, said the country “remains committed to the recent ceasefire understanding, taking necessary steps towards de-escalation, regional stability,” according to Reuters.

“Pakistan rejects provocative and inflammatory assertions by the indian prime minister,” Reuters quoted the country's foreign ministry, which also said that “we hope that India will prioritise regional stability, well-being of its citizens over narrow and politically motivated jingoism".

Any future aggression will also be met with full resolve, the Pakistani foreign ministry said.

Narendra Modi on Tuesday also issued a strong warning to Pakistan, saying it will bite the dust if it allows even one more terror attack in India. Addressing air force personnel at the Adampur air base in Punjab, Modi said, "India is always with peace, but it is always ready to make the enemy bite the dust if it is attacked.

India-Pakistan ceasefire
After four days of an intense exchange of fire, India and Pakistan on Saturday reached a ceasefire ‘understanding’ to immediately stop all firing and military action on land, air and sea. The ceasefire, which was first announced by US President Donald Trump on Saturday, was violated by Pakistan hours later with drones being intercepted over parts of Jammu, Srinagar, Punjab and Rajasthan.

On Monday as well, drones were spotted over Jammu and Punjab shortly after PM Modi's speech, however, after initial alert, the situation remained peaceful at the border areas through the night.

The military confrontation erupted after Islamabad launched drones and missiles towards the Indian territory, responding to the Operation Sindoor military strikes carried out by New Delhi on nine terror infrastructures in Pakistan and Pakistan-occupied Kashmir on May 7.

Operation Sindoor was launched in retaliation to the April 22 terror attack in Jammu and Kashmir's Pahalgam in which terrorists found to have links with Pakistan killed 26 civilians.

"""

In [29]:
# Summarize your sample text
generated_summary = summarizer(article, max_length=60, min_length=20, do_sample=False)[0]['summary_text']

# Simplify
simple_summary = simplify_text(generated_summary, simplify_dict)

# Classify topic
result = classifier(simple_summary, candidate_labels)
top_topic = result["labels"][0]

# Bias analysis
bias_result = detect_bias(simple_summary)

# Readability
original_grade = textstat.flesch_kincaid_grade(generated_summary)
simple_grade = textstat.flesch_kincaid_grade(simple_summary)

# Output
print("Model Summary:", generated_summary)
print("Simplified Summary:", simple_summary)
print("Topic:", top_topic)
print("Bias:", bias_result)
print("Original Grade:", original_grade)
print("Simplified Grade:", simple_grade)


Model Summary: Prime Minister Modi on Monday addressed the nation and lauded the Indian armed forces for successfully carrying out Operation Sindoor. Pakistan's foreign ministry, reacting to Modi's address, said the country “remains committed to the recent ceasefire understanding”
Simplified Summary: Prime Minister Modi on Monday addressed the nation and lauded the Indian armed forces for successfully carrying out Operation Sindoor. Pakistan's foreign ministry, reacting to Modi's address, said the country “remains committed to the recent ceasefire understanding”
Topic: politics
Bias: Neutral
Original Grade: 15.543918918918923
Simplified Grade: 15.543918918918923


# again another one:

In [7]:
!pip install transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Load pretrained T5 model fine-tuned for grammar correction
model_name = "vennify/t5-base-grammar-correction"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create the pipeline
feedback_generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Sample student essay
student_essay = """
The student have went to the school but he not bring his books. 
It were a important day and he miss the class test because he did not know.
"""

# Generate grammar-corrected version
prompt = f"grammar: {student_essay.strip()}"
response = feedback_generator(prompt, max_length=512, clean_up_tokenization_spaces=True)[0]['generated_text']

print("✍️ Corrected Essay:\n")
print(response)


tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

Device set to use mps:0


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

✍️ Corrected Essay:

The student went to school but he did not bring his books. It was an important day and he missed the class test because he did not know.


In [12]:
student_essay = """
Education speeds up effective learning and instils values, information, skills and beliefs. A person’s life becomes better and more peaceful as a result of education. The teaching of writing and reading is the first stage in education. People become conscious and literate through education. It makes it easier for people to find work and undoubtedly improves their standard of living. Additionally, it enhances and hones a person’s communication abilities. It teaches someone to make practical use of the resources. The significance of education in advancing knowledge in society is one of its significant features. When a person is educated, knowledge is transferred from one generation to the next. Not one person, but many people are educated because of one. 

As one’s knowledge base grows and their technical proficiency improves, education strengthens a person intellectually, mentally and socially. In the business and academic worlds, it helps in improving their position. It serves as a useful tool for all stages of life. In the cutting-edge technological environment, education is crucial. Unlike in the past, when only the wealthy could afford to send their children to school and receive training, education is not as difficult or expensive. In the twenty-first century, there are numerous strategies to raise educational standards. In today’s modernised period, the requirements for receiving an education have altered completely.

Nowadays, anyone, regardless of age, can pursue an education. If a person’s thinking is not limited, their age will never be a barrier. The possibility of homeschooling has been made available in some curricula. Universities around the globe are starting a variety of distance learning programmes. Following high school, we can pursue both a job and further education through remote learning programmes. To make the courses available to everyone, the academic price has also been made affordable.

Governmental and non-governmental agencies organise a variety of events where teachers visit a community and impart knowledge to students. In order to assist someone become an educated person, parents and instructors play a crucial part in their lives. Through education people’s mindset is improved which leads to the removal of significant social barriers. It advances not just societal and economic progress but also personal advancements.

Any country’s greatest advantage is its educated population. Through them, a nation improves because education breaks down mindset barriers, imparts knowledge and information, and develops people’s listening skills and manners. It gives a person a higher standard of living and equips them to deal with issues at the local, state, federal, and worldwide levels. Education promotes self-reliance, mental stability, and financial security. It improves self-assurance and instils confidence in a person, which is one of the best qualities of success.
"""

# Generate grammar-corrected version
prompt = f"grammar: {student_essay.strip()}"
response = feedback_generator(prompt, max_length=len(student_essay), clean_up_tokenization_spaces=True)[0]['generated_text']

print("✍️ Corrected Essay:\n")
print(response)


✍️ Corrected Essay:

Education speeds up effective learning and instils values, information, skills and beliefs. A person’s life becomes better and more peaceful as a result of education. It makes it easier for people to find work and undoubtedly improves their standard of living. Additionally, it enhances and hones a person’s communication abilities. It helps them to make practical use of the resources.
